In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.history import HISTORY_FEATURES
from src.modeling import make_features, build_model, evaluate_scores

In [2]:
history_features = pd.read_parquet(
    project_root / "data/processed/lab03_history_features.parquet"
)

datasets = {}

for split_name in ["train", "validation"]:
    base = pd.read_parquet(
        project_root / f"data/processed/lab02_{split_name}.parquet"
    )

    enriched = base.merge(
        history_features,
        on="source_row",
        how="left",
        sort=False,
        validate="one_to_one",
    )

    assert enriched["source_row"].equals(base["source_row"])
    assert enriched[HISTORY_FEATURES].notna().all().all()

    datasets[split_name] = enriched

train = datasets["train"]
validation = datasets["validation"]

In [3]:
extra_features = tuple(HISTORY_FEATURES)

model = build_model(
    extra_numeric_features=extra_features,
)

with threadpool_limits(limits=2):
    model.fit(
        make_features(train, extra_numeric_features=extra_features),
        train["is_laundering"],
    )

    scores = model.predict_proba(
        make_features(
            validation,
            extra_numeric_features=extra_features,
        )
    )[:, 1]

In [4]:
review_table = validation.reset_index(drop=True).copy()

review_table["score"] = scores
review_table["day"] = review_table["timestamp"].dt.normalize()
review_table["tie_breaker"] = (
    np.random.default_rng(42).random(len(review_table))
)

review_table = (
    review_table
    .sort_values(
        ["day", "score", "tie_breaker"],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

review_table["daily_rank"] = (
    review_table.groupby("day", sort=False).cumcount() + 1
)

review_table["selected_for_review"] = (
    review_table["daily_rank"] <= 50
)

In [5]:
review_table.loc[
    review_table["selected_for_review"],
    "payment_format",
].value_counts()

payment_format
ACH            100
Bitcoin          0
Cash             0
Cheque           0
Credit Card      0
Wire             0
Name: count, dtype: int64

In [6]:
selected = review_table["selected_for_review"]
positive = review_table["is_laundering"].eq(1)

review_table["outcome"] = pd.Categorical(
    np.select(
        [
            selected & positive,
            selected & ~positive,
            ~selected & positive,
        ],
        [
            "caught_laundering",
            "false_alert",
            "missed_laundering",
        ],
        default="not_reviewed_legitimate",
    )
)

display(review_table["outcome"].value_counts())

outcome
not_reviewed_legitimate    964440
missed_laundering             984
caught_laundering              52
false_alert                    48
Name: count, dtype: int64

In [7]:
metrics = evaluate_scores(validation, scores, daily_budget=50)

assert review_table["source_row"].is_unique

assert (
    review_table["selected_for_review"].sum()
    == metrics["alerts"]
    == 100
)

assert (
    review_table["outcome"].eq("caught_laundering").sum()
    == metrics["laundering_caught"]
    == 52
)

assert (
    review_table.groupby("day")["selected_for_review"]
    .sum()
    .eq(50)
    .all()
)

In [8]:
display(
    review_table.loc[
        review_table["selected_for_review"],
        [
            "source_row",
            "timestamp",
            "payment_format",
            "amount_paid",
            "payment_currency",
            "score",
            "daily_rank",
            "outcome",
            *HISTORY_FEATURES,
        ],
    ].head(10)
)

,source_row,timestamp,payment_format,amount_paid,payment_currency,score,daily_rank,outcome,sender_in_count_24h,receiver_in_count_24h,sender_out_count_24h,receiver_out_count_24h
0,3123393,2022-09-07 15:19:00,ACH,13501.86,Euro,0.549943,1,caught_laundering,0,3,0,12
1,3123423,2022-09-07 12:46:00,ACH,15639.47,Euro,0.546895,2,caught_laundering,12,0,16,0
2,3303599,2022-09-07 13:48:00,ACH,12655.16,US Dollar,0.541505,3,caught_laundering,0,4,0,8
3,2821101,2022-09-07 13:11:00,ACH,28525.51,Saudi Riyal,0.536379,4,caught_laundering,3,5,2,4
4,3201413,2022-09-07 15:54:00,ACH,12268.52,Euro,0.516124,5,caught_laundering,3,3,0,7
5,3625784,2022-09-07 18:02:00,ACH,49946.10,Saudi Riyal,0.480463,6,caught_laundering,5,9,1,1
6,2631782,2022-09-07 05:00:00,ACH,26259.53,Saudi Riyal,0.480314,7,caught_laundering,5,2,0,12
7,2192349,2022-09-07 10:06:00,ACH,10162.68,US Dollar,0.477702,8,caught_laundering,0,1,0,16
8,3133000,2022-09-07 14:28:00,ACH,16658.75,Euro,0.474880,9,caught_laundering,0,2,0,13
9,3522325,2022-09-07 13:12:00,ACH,17645.46,Euro,0.466751,10,false_alert,2,3,0,13


In [9]:
review_table.to_parquet(
    project_root / "data/processed/lab04_review_table.parquet",
    index=False,
)

### Review-table setup

We reproduced the Lab 3 history model and selected 50 transactions
per validation day.

The 100 reviews contain 52 laundering transactions and 48 legitimate
transactions. Another 984 laundering transactions are outside the budget.

We saved scores, daily ranks, selections, and outcomes for case inspection.

In [10]:
review_table = pd.read_parquet(
    project_root / "data/processed/lab04_review_table.parquet"
)

false_alerts = (
    review_table.loc[review_table["outcome"].eq("false_alert")]
    .sort_values(["score", "source_row"], ascending=[False, True])
    .head(5)
)

misses = (
    review_table.loc[review_table["outcome"].eq("missed_laundering")]
    .sample(n=5, random_state=42)
)

caught = (
    review_table.loc[review_table["outcome"].eq("caught_laundering")]
    .sample(n=5, random_state=42)
)

cases = pd.concat(
    [false_alerts, misses, caught],
    ignore_index=True,
)

assert len(cases) == 15
assert cases["source_row"].is_unique

In [11]:
cases

,timestamp,amount_received,receiving_currency,amount_paid,payment_currency,payment_format,is_laundering,source_row,sender_in_count_24h,receiver_in_count_24h,sender_out_count_24h,receiver_out_count_24h,score,day,tie_breaker,daily_rank,selected_for_review,outcome
0,2022-09-08 12:57:00,36401.32,Saudi Riyal,36401.32,Saudi Riyal,ACH,0,3625785,4,4,1,2,0.516926,2022-09-08,0.144074,1,True,false_alert
1,2022-09-08 16:52:00,70198.56,Saudi Riyal,70198.56,Saudi Riyal,ACH,0,4078643,4,5,2,2,0.491842,2022-09-08,0.945156,2,True,false_alert
2,2022-09-08 12:45:00,17214.01,US Dollar,17214.01,US Dollar,ACH,0,3990907,0,2,0,10,0.472388,2022-09-08,0.630919,4,True,false_alert
3,2022-09-07 13:12:00,17645.46,Euro,17645.46,Euro,ACH,0,3522325,2,3,0,13,0.466751,2022-09-07,0.760178,10,True,false_alert
4,2022-09-07 13:30:00,14877.17,US Dollar,14877.17,US Dollar,ACH,0,3529509,7,2,0,10,0.456298,2022-09-07,0.852361,13,True,false_alert
5,2022-09-08 14:01:00,633401.43,Yen,633401.43,Yen,ACH,1,3586641,1,0,7,0,0.048344,2022-09-08,0.386645,1286,False,missed_laundering
6,2022-09-07 10:17:00,1320.14,Euro,1320.14,Euro,Credit Card,1,3464995,0,1,11216,0,0.001216,2022-09-07,0.371437,84674,False,missed_laundering
7,2022-09-08 10:22:00,1658.03,Euro,1658.03,Euro,ACH,1,1237949,3,0,4,0,0.013405,2022-09-08,0.325576,5590,False,missed_laundering
8,2022-09-07 23:01:00,36071.74,Yen,36071.74,Yen,Cash,1,3719577,0,6,1515,2,0.001243,2022-09-07,0.604996,73692,False,missed_laundering
9,2022-09-07 05:56:00,3345.22,US Dollar,3345.22,US Dollar,ACH,1,3372073,0,2,0,0,0.012270,2022-09-07,0.482748,6506,False,missed_laundering


In [12]:
import importlib
import src.data

importlib.reload(src.data)

from src.data import read_source_rows

endpoint_columns = [
    "sender_bank",
    "sender_account",
    "receiver_bank",
    "receiver_account",
]

endpoints = read_source_rows(
    project_root / "data/raw/archive/HI-Small_Trans.csv",
    source_rows=cases["source_row"],
    columns=["timestamp", *endpoint_columns],
)

endpoints["timestamp"] = pd.to_datetime(
    endpoints["timestamp"],
    format="%Y/%m/%d %H:%M",
    errors="raise",
)

case_table = cases.merge(
    endpoints,
    on=["source_row", "timestamp"],
    how="left",
    sort=False,
    validate="one_to_one",
)

assert len(case_table) == 15
assert case_table[endpoint_columns].notna().all().all()
assert case_table["source_row"].equals(cases["source_row"])

In [13]:
inspection_columns = [
    "source_row",
    "payment_format",
    "amount_paid",
    "payment_currency",
    "score",
    "daily_rank",
    *HISTORY_FEATURES,
]

for outcome in [
    "false_alert",
    "missed_laundering",
    "caught_laundering",
]:
    print(outcome)

    display(
        case_table.loc[
            case_table["outcome"].eq(outcome),
            inspection_columns,
        ]
    )

false_alert


,source_row,payment_format,amount_paid,payment_currency,score,daily_rank,sender_in_count_24h,receiver_in_count_24h,sender_out_count_24h,receiver_out_count_24h
0,3625785,ACH,36401.32,Saudi Riyal,0.516926,1,4,4,1,2
1,4078643,ACH,70198.56,Saudi Riyal,0.491842,2,4,5,2,2
2,3990907,ACH,17214.01,US Dollar,0.472388,4,0,2,0,10
3,3522325,ACH,17645.46,Euro,0.466751,10,2,3,0,13
4,3529509,ACH,14877.17,US Dollar,0.456298,13,7,2,0,10


missed_laundering


,source_row,payment_format,amount_paid,payment_currency,score,daily_rank,sender_in_count_24h,receiver_in_count_24h,sender_out_count_24h,receiver_out_count_24h
5,3586641,ACH,633401.43,Yen,0.048344,1286,1,0,7,0
6,3464995,Credit Card,1320.14,Euro,0.001216,84674,0,1,11216,0
7,1237949,ACH,1658.03,Euro,0.013405,5590,3,0,4,0
8,3719577,Cash,36071.74,Yen,0.001243,73692,0,6,1515,2
9,3372073,ACH,3345.22,US Dollar,0.012270,6506,0,2,0,0


caught_laundering


,source_row,payment_format,amount_paid,payment_currency,score,daily_rank,sender_in_count_24h,receiver_in_count_24h,sender_out_count_24h,receiver_out_count_24h
10,2358619,ACH,18763.12,US Dollar,0.400277,26,2,2,0,9
11,2641334,ACH,15471.36,US Dollar,0.332564,26,6,4,0,1
12,2548024,ACH,17441.40,US Dollar,0.292805,42,0,2,0,2
13,3123421,ACH,826736.97,Rupee,0.455447,15,11,0,14,0
14,3123485,ACH,51315.83,Saudi Riyal,0.328888,28,4,4,3,13


In [14]:
case_table.to_parquet(
    project_root / "data/processed/lab04_cases.parquet",
    index=False,
)

### Case inspection

- The sampled misses rank far below the daily cutoff.
- All 100 validation reviews are ACH. ACH contributes most training
  laundering examples, despite being a minority of transactions.
- Amount comparisons require matching currencies. Sampled USD false
  alerts and caught cases have overlapping amounts.

**Hypothesis**:
The model may struggle with non-ACH laundering because those payment
types have relatively few positive training examples.

In [15]:
import importlib
import src.modeling

importlib.reload(src.modeling)

from src.modeling import make_features, build_model, evaluate_scores

configuration = {
    "extra_numeric_features": tuple(HISTORY_FEATURES),
    "excluded_features": ("payment_format",),
}

X_train = make_features(train, **configuration)
X_validation = make_features(validation, **configuration)

assert "payment_format" not in X_train
assert X_train.shape[1] == 10

model_without_format = build_model(**configuration)

with threadpool_limits(limits=2):
    model_without_format.fit(
        X_train,
        train["is_laundering"],
    )

    scores_without_format = model_without_format.predict_proba(
        X_validation
    )[:, 1]

metrics_without_format = evaluate_scores(
    validation,
    scores_without_format,
    daily_budget=50,
)

In [16]:
reference = pd.read_csv(
    project_root / "reports/lab_03_history_metrics.csv",
    index_col=0,
).loc["with_history"]

metric_columns = [
    "average_precision",
    "alerts",
    "laundering_caught",
    "precision_at_budget",
    "recall_at_budget",
]

comparison = pd.DataFrame({
    "with_format": reference[metric_columns].to_dict(),
    "without_format": metrics_without_format,
}).T

display(comparison.round(4))

,average_precision,alerts,laundering_caught,precision_at_budget,recall_at_budget
with_format,0.1338,100.0,52.0,0.52,0.0502
without_format,0.0301,100.0,19.0,0.19,0.0183


In [17]:
ranked = validation[
    ["timestamp", "payment_format", "is_laundering"]
].reset_index(drop=True).copy()

ranked["score"] = scores_without_format
ranked["day"] = ranked["timestamp"].dt.normalize()
ranked["tie_breaker"] = (
    np.random.default_rng(42).random(len(ranked))
)

selected_without_format = (
    ranked.sort_values(
        ["day", "score", "tie_breaker"],
        ascending=[True, False, True],
    )
    .groupby("day", sort=False)
    .head(50)
)

assert len(selected_without_format) == 100

assert (
    selected_without_format["is_laundering"].sum()
    == metrics_without_format["laundering_caught"]
)

In [18]:
previous_reviews = pd.read_parquet(
    project_root / "data/processed/lab04_review_table.parquet"
)

selected_with_format = previous_reviews.loc[
    previous_reviews["selected_for_review"]
]

In [19]:
payment_formats = sorted(
    validation["payment_format"].dropna().unique()
)

def summarize_reviews(selected, *, payment_formats):
    return (
        selected.groupby("payment_format", observed=True)
        .agg(
            reviews=("is_laundering", "size"),
            caught=("is_laundering", "sum"),
        )
        .reindex(payment_formats, fill_value=0)
    )

coverage = pd.concat({
    "with_format": summarize_reviews(
        selected_with_format,
        payment_formats=payment_formats,
    ),
    "without_format": summarize_reviews(
        selected_without_format,
        payment_formats=payment_formats,
    ),
}, axis=1)

display(coverage)

with_format        without_format       
                   reviews caught        reviews caught
payment_format                                         
ACH                    100     52             46     19
Bitcoin                  0      0              0      0
Cash                     0      0              5      0
Cheque                   0      0             29      0
Credit Card              0      0             18      0
Wire                     0      0              2      0

In [20]:
comparison.to_csv(
    project_root / "reports/lab_04_ablation_metrics.csv"
)

coverage.to_csv(
    project_root / "reports/lab_04_ablation_by_format.csv"
)

### Removing payment format

- Average precision changed from 0.1338 to 0.0301.
- Laundering detections changed from 52 to 19, within 100 reviews.
- Non-ACH reviews changed from zero to 54, catching 0 positives.

**Interpretation**:
Payment format provides useful information under our current model and review budget. We’ll retain it.